In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy import signal
import plotly.express as px
from scipy.signal import find_peaks
import os
from pydub import AudioSegment

import params as yamnet_params
import yamnet as yamnet_model
import soundfile as sf
import resampy

# custom functions to perfomr Leq and db sum
def sum_levels(levels):
    l = np.array(levels)
    return 10*np.log10(np.sum(np.power(10,l/10)))

def leq(levels):
    l = np.array(levels)
    return 10*np.log10(np.mean(np.power(10,l/10)))

c:\Users\scjaa\anaconda3\envs\tensorflow_gpu_11_2\lib\site-packages\pydub\utils.py:170: RuntimeWarning: Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work
  warn("Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work", RuntimeWarning)


In [2]:
# opne csv file
csv_file = r"\\192.168.205.117\AAC_Server\PUERTOS\NOISEPORT\20231211_SANTUR\5-Resultados\P1_CONTENEDORES\SPL\leq_levels_P1_CONTENEDORES_v1_1_test.csv"
df = pd.read_csv(csv_file)
title = csv_file.split("\\")[-3]

# get the audiomoth folder
audiomoth_folder = csv_file.replace("5-Resultados","3-Medidas").replace("SPL","AUDIOMOTH")
# remove last item from audio moth folder
audiomoth_folder = "\\".join(audiomoth_folder.split("\\")[:-1])

output_folder = csv_file.split("\\")[:-1]
# join the folder
output_folder = "\\".join(output_folder)
print(output_folder)
# add 
if not os.path.exists(output_folder):
    print("Creating folder")
else:
    print("Folder already exists")

# # # chagen the filename to the full path
df['filename'] = df['filename'].apply(lambda x: os.path.join(audiomoth_folder, x))
df

\\192.168.205.117\AAC_Server\PUERTOS\NOISEPORT\20231211_SANTUR\5-Resultados\P1_CONTENEDORES\SPL
Folder already exists


,LA,LC,LZ,LAmax,LAmin,filename,date
0,72.07,79.36,79.97,73.09,70.91,\\192.168.205.117\AAC_Server\PUERTOS\NOISEPORT...,2023-12-11 14:04:20
1,72.40,79.43,79.86,73.70,71.27,\\192.168.205.117\AAC_Server\PUERTOS\NOISEPORT...,2023-12-11 14:04:21
2,72.16,78.31,78.61,73.06,71.27,\\192.168.205.117\AAC_Server\PUERTOS\NOISEPORT...,2023-12-11 14:04:22
3,72.38,78.21,78.32,73.86,70.97,\\192.168.205.117\AAC_Server\PUERTOS\NOISEPORT...,2023-12-11 14:04:23
4,71.70,77.32,77.41,72.21,71.26,\\192.168.205.117\AAC_Server\PUERTOS\NOISEPORT...,2023-12-11 14:04:24
...,...,...,...,...,...,...,...
714003,70.90,77.58,77.77,73.48,68.05,\\192.168.205.117\AAC_Server\PUERTOS\NOISEPORT...,2023-12-20 06:21:15
714004,70.41,76.66,76.79,73.97,67.94,\\192.168.205.117\AAC_Server\PUERTOS\NOISEPORT...,2023-12-20 06:21:16
714005,67.92,74.28,74.41,69.86,66.57,\\192.168.205.117\AAC_Server\PUERTOS\NOISEPORT...,2023-12-20 06:21:17
714006,68.78,74.67,74.79,71.63,66.69,\\192.168.205.117\AAC_Server\PUERTOS\NOISEPORT...,2023-12-20 06:21:18


In [3]:
# convert date column in datetime
df['date'] = pd.to_datetime(df['date'])

duration = df['date'].iloc[-1] - df['date'].iloc[0]
duration = duration.total_seconds()
print(f"Duration: {duration} seconds, {duration/60} minutes, {duration/3600} hours, {duration/3600/24} days")

la = df['LA'].values
print('\nMax:', np.max(la).round(2))
print('Min:', np.min(la).round(2))

print('\nMediana:', np.median(la).round(2))
print('Promedio:', leq(la).round(2))
print('Standard deviation:', np.std(la).round(2))

print('\nPercentil 98:', np.quantile(la, 0.99).round(2)) # percentil 98 means that 98% of the data is below this value

Duration: 749819.0 seconds, 12496.983333333334 minutes, 208.28305555555556 hours, 8.678460648148148 days

Max: 96.9
Min: 55.77

Mediana: 67.47
Promedio: 72.69
Standard deviation: 5.5

Percentil 98: 82.8


In [6]:
def save_clips_from_peaks(df, output_folder, sampling_rate):
    csv_output_folder = output_folder
    # if output folder does not exist, create it
    output_folder = os.path.join(output_folder, "peak_clips")
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
    
    for index, row in df.iterrows():
        audio_file = row['filename']
        start_time_seconds = row['start_time']
        end_time_seconds = row['end_time']

        #loading audio file
        wav_data, sr = sf.read(audio_file, dtype='int16')
        waveform = wav_data / 32768.0  # Convert to [-1.0, +1.0]

        
        print(f"Processing audio file: {audio_file}")
        print(f"Starting time: {start_time_seconds} seconds, Ending time: {end_time_seconds} seconds")
        
        # milliseconds
        start_time_ms = int(start_time_seconds * 1000)
        end_time_ms = int(end_time_seconds * 1000)
        print(f"Starting time milliseconds: {start_time_ms}, Ending time milliseconds: {end_time_ms}")


        # audio = AudioSegment.from_file(audio_file)
        clip = waveform[start_time_ms : end_time_ms]
        waveform = waveform.astype('float32')

        if len(waveform.shape) > 1:
            waveform = np.mean(waveform, axis=1)
        if sr != params.sample_rate:
            waveform = resampy.resample(waveform, sr, params.sample_rate)
        
        scores, embeddings, spectrogram = yamnet(waveform)
        prediction = np.mean(scores, axis=0)
        top3_i = np.argsort(prediction)[::-1][:3]

        print()
        print(clip, ':\n' +
          '\n'.join('  {:12s}: {:.3f}'.format(yamnet_classes[i], prediction[i])
                    for i in top3_i))
        print()

        # save a dataframe with the filename, start time, end time, and the top 3 classes
        filename = audio_file.split("\\")[-1]
        df_clip = pd.DataFrame({
            'filename': filename,
            'start_time': [start_time_seconds],
            'end_time': [end_time_seconds],
            'predicitons': [', '.join([yamnet_classes[i] for i in top3_i])],
            'probabilities': [', '.join([str(prediction[i]) for i in top3_i])],
        })

        # save the dataframe to a csv file
        csv_filename = f'peak_predictions_L50_{title}.csv'
        csv_path = os.path.join(csv_output_folder, csv_filename)
        # save to csv with the header only if the file does not exist
        df_clip.to_csv(csv_path, mode='a', header=not os.path.exists(csv_path), index=False)
        print(f"Saving predictions to {csv_path}")

        print(f"Filename: {filename}")
        clip_path = os.path.join(output_folder, f"{filename}_peak_{yamnet_classes[top3_i[0]]}.wav")
        print(f"Saving clip {clip_path}")
        print(f"Clip duration: {len(clip)} milliseconds")  # debugging 

        # using sf.write to save the clip
        sf.write(clip_path, clip, sampling_rate)



params = yamnet_params.Params()
yamnet = yamnet_model.yamnet_frames_model(params)
yamnet.load_weights('yamnet.h5')
yamnet_classes = yamnet_model.class_names('yamnet_class_map.csv')

window_size = 30  # seconds
prominence = 1
width = 1

df['LA_median'] = df['LA'].rolling(window=window_size, min_periods=1).quantile(0.5) + 10

# go through each day
start_date = df['date'].dt.normalize().iloc[0]
end_date = df['date'].dt.normalize().iloc[-1]
current_date = start_date

peaks_num = []
while current_date <= end_date:
    daily_data = df[(df['date'] >= current_date) & (df['date'] < current_date + pd.Timedelta(days=1))]
    
    #dynamic threshold by filtering data points
    above_threshold = daily_data[daily_data['LA'] > daily_data['LA_median']]
    
    #find peaks in the filtered data
    if not above_threshold.empty:
        peaks, properties = find_peaks(above_threshold['LA'], prominence=prominence, width=width)
        peak_filenames = above_threshold.iloc[peaks]['filename'].values
        start_times = np.round(properties['left_ips'] ,2)
        end_times = np.round(properties['right_ips'], 2)
        
        #df for each peak
        peaks_df = pd.DataFrame({
            'filename': peak_filenames,
            'start_time': start_times,
            'end_time': end_times
        })

        # take the sr from the first audio file
        audio_file = peak_filenames[0]
        audio = AudioSegment.from_file(audio_file)
        sampling_rate = audio.frame_rate
        
        # SAVING CLIPS FROM PEAKS
        save_clips_from_peaks(peaks_df, output_folder, sampling_rate)    

Processing audio file: \\192.168.205.117\AAC_Server\PUERTOS\NOISEPORT\20231211_SANTUR\3-Medidas\P1_CONTENEDORES\AUDIOMOTH\20231211_140420.WAV
Starting time: 3.95 seconds, Ending time: 6.5 seconds
Starting time milliseconds: 3950, Ending time milliseconds: 6500

[ 0.01315308  0.0241394   0.03451538 ... -0.03085327 -0.02420044
 -0.02230835] :
  Vehicle     : 0.489
  Aircraft    : 0.160
  Fixed-wing aircraft, airplane: 0.122

Saving predictions to \\192.168.205.117\AAC_Server\PUERTOS\NOISEPORT\20231211_SANTUR\5-Resultados\P1_CONTENEDORES\SPL\peak_predictions_L50_P1_CONTENEDORES.csv
Filename: 20231211_140420.WAV
Saving clip \\192.168.205.117\AAC_Server\PUERTOS\NOISEPORT\20231211_SANTUR\5-Resultados\P1_CONTENEDORES\SPL\peak_clips\20231211_140420.WAV_peak_Vehicle.wav
Clip duration: 2550 milliseconds
Processing audio file: \\192.168.205.117\AAC_Server\PUERTOS\NOISEPORT\20231211_SANTUR\3-Medidas\P1_CONTENEDORES\AUDIOMOTH\20231211_140420.WAV
Starting time: 8.66 seconds, Ending time: 10.53 secon

KeyboardInterrupt: 